# Notebook 5 — Future Disruption Prediction
User enters shipment details → system fetches live weather → runs Isolation Forest → LSTM → XGBoost → prints full prediction report

In [1]:
import pandas as pd
import numpy as np
import requests
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import load_model

DATA_DIR   = '../datasets/'
MODELS_DIR = '../models/'
print('Imports done.')

Imports done.


In [2]:
# Load all saved models
xgb_model   = joblib.load(MODELS_DIR + 'xgboost_model.pkl')
lstm_model  = load_model(MODELS_DIR + 'lstm_model.h5')
iso_forest  = joblib.load(MODELS_DIR + 'isolation_forest.pkl')
scaler      = joblib.load(MODELS_DIR + 'scaler.pkl')
le_transport = joblib.load(MODELS_DIR + 'le_transport.pkl')
le_product   = joblib.load(MODELS_DIR + 'le_product.pkl')
le_city      = joblib.load(MODELS_DIR + 'le_city.pkl')
le_dest      = joblib.load(MODELS_DIR + 'le_dest.pkl')

with open(MODELS_DIR + 'feature_list.json') as f:
    XGB_FEATURES = json.load(f)

print('All models and encoders loaded. ✅')
print(f'XGBoost features: {len(XGB_FEATURES)}')

All models and encoders loaded. ✅
XGBoost features: 25


In [3]:
# Load lookup tables from CSVs
d3 = pd.read_csv(DATA_DIR + 'dataset3_news_sentiment_india.csv', parse_dates=['date'])
d4 = pd.read_csv(DATA_DIR + 'dataset4_supplier_risk_india.csv')
d5 = pd.read_csv(DATA_DIR + 'dataset5_historical_disruptions_india.csv')
merged = pd.read_csv(DATA_DIR + 'merged_dataset.csv')

# D3: city + month average sentiment lookup
d3['month'] = d3['date'].dt.month
sentiment_lookup = d3.groupby(['affected_city','month']).agg(
    avg_sentiment=('sentiment_score','mean'),
    avg_disruption_signal=('disruption_signal','mean')
).reset_index()

# D5: city historical disruption count average
city_hist_avg = merged.groupby('supplier_city')['historical_disruption_count'].mean().to_dict()

# City rolling delay averages
city_rolling = merged.groupby('supplier_city').agg(
    avg_rolling7 =('rolling_7d_delay','mean'),
    avg_rolling14=('rolling_14d_delay','mean'),
    avg_lag1     =('lag1_delay','mean'),
    avg_lag2     =('lag2_delay','mean')
).to_dict('index')

print('Lookup tables built. ✅')

Lookup tables built. ✅


In [4]:
# City coordinates for weather forecast
CITY_COORDS = {
    'Mumbai':    (19.08, 72.88), 'Delhi':     (28.61, 77.21),
    'Chennai':   (13.08, 80.27), 'Kolkata':   (22.57, 88.36),
    'Bangalore': (12.97, 77.59), 'Hyderabad': (17.38, 78.49),
    'Pune':      (18.52, 73.86), 'Ahmedabad': (23.03, 72.58),
    'Jaipur':    (26.91, 75.79), 'Lucknow':   (26.85, 80.95),
    'Surat':     (21.17, 72.83), 'Kanpur':    (26.46, 80.33),
    'Nagpur':    (21.15, 79.08), 'Indore':    (22.72, 75.86),
    'Bhopal':    (23.26, 77.41),
}

LSTM_FEATURES = [
    'distance_km','promised_delivery_days','quantity_units',
    'rainfall_mm','temperature_celsius','wind_speed_kmh',
    'severity_score','weather_anomaly_score',
    'sentiment_score','disruption_signal',
    'composite_risk_score','delivery_reliability_score',
    'regional_risk_index','historical_disruption_count',
    'rolling_7d_delay','rolling_14d_delay','lag1_delay','lag2_delay',
    'transport_mode_enc','product_category_enc','supplier_city_enc'
]
SEQ_LEN = 14

print('Valid supplier cities:', list(CITY_COORDS.keys()))
print('Valid transport modes:', list(le_transport.classes_))
print('Valid product categories:', list(le_product.classes_))
print('Valid supplier names:', list(d4['supplier_name'].values))

Valid supplier cities: ['Mumbai', 'Delhi', 'Chennai', 'Kolkata', 'Bangalore', 'Hyderabad', 'Pune', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Surat', 'Kanpur', 'Nagpur', 'Indore', 'Bhopal']
Valid transport modes: ['Air', 'Rail', 'Road', 'Road+Rail']
Valid product categories: ['Agricultural Goods', 'Automotive Parts', 'Chemical Compounds', 'Electronics', 'FMCG', 'IT Hardware', 'Industrial Machinery', 'Pharmaceuticals', 'Raw Materials', 'Textiles']
Valid supplier names: ['Tata Suppliers Pvt Ltd', 'Reliance Logistics', 'Infosys Supply Co', 'Mahindra Parts Ltd', 'Bajaj Materials', 'Wipro Procurement', 'HCL Vendors', 'Larsen & Toubro Supply', 'BHEL Components', 'NTPC Materials', 'ONGC Supplies', 'Indian Oil Logistics', 'Steel Authority India', 'Coal India Suppliers', 'GAIL Vendors', 'Tata Suppliers Pvt Ltd', 'Reliance Logistics', 'Infosys Supply Co', 'Mahindra Parts Ltd', 'Bajaj Materials', 'Wipro Procurement', 'HCL Vendors', 'Larsen & Toubro Supply', 'BHEL Components', 'NTPC Materials', 'ONGC Supp

In [5]:
# Helper functions — with input validation and auto-correction

import difflib

# Valid values from encoders
VALID_CITIES     = list(CITY_COORDS.keys())
VALID_TRANSPORTS = list(le_transport.classes_)
VALID_PRODUCTS   = list(le_product.classes_)
VALID_SUPPLIERS  = list(d4['supplier_name'].values)

def fuzzy_match(user_input, valid_list):
    """Case-insensitive exact match first, then closest fuzzy match."""
    # Try case-insensitive exact match
    for v in valid_list:
        if v.lower() == user_input.lower().strip():
            return v, True
    # Try fuzzy match — finds closest option
    matches = difflib.get_close_matches(user_input.strip(), valid_list, n=1, cutoff=0.5)
    if matches:
        return matches[0], True
    return None, False

def validate_inputs(origin_city, destination_city, product_category,
                    transport_mode, quantity_units, order_date, supplier_name):
    errors  = []
    fixed   = {}

    # Origin city
    match, ok = fuzzy_match(origin_city, VALID_CITIES)
    if ok:
        if match != origin_city:
            print(f'  Auto-corrected origin city: "{origin_city}" → "{match}"')
        fixed['origin_city'] = match
    else:
        errors.append(f'Origin city "{origin_city}" not recognised.\n    Valid options: {VALID_CITIES}')

    # Destination city
    match, ok = fuzzy_match(destination_city, VALID_CITIES)
    if ok:
        if match != destination_city:
            print(f'  Auto-corrected destination city: "{destination_city}" → "{match}"')
        fixed['destination_city'] = match
    else:
        errors.append(f'Destination city "{destination_city}" not recognised.\n    Valid options: {VALID_CITIES}')

    # Product category
    match, ok = fuzzy_match(product_category, VALID_PRODUCTS)
    if ok:
        if match != product_category:
            print(f'  Auto-corrected product category: "{product_category}" → "{match}"')
        fixed['product_category'] = match
    else:
        errors.append(f'Product category "{product_category}" not recognised.\n    Valid options: {VALID_PRODUCTS}')

    # Transport mode
    match, ok = fuzzy_match(transport_mode, VALID_TRANSPORTS)
    if ok:
        if match != transport_mode:
            print(f'  Auto-corrected transport mode: "{transport_mode}" → "{match}"')
        fixed['transport_mode'] = match
    else:
        errors.append(f'Transport mode "{transport_mode}" not recognised.\n    Valid options: {VALID_TRANSPORTS}')

    # Supplier name
    match, ok = fuzzy_match(supplier_name, VALID_SUPPLIERS)
    if ok:
        if match != supplier_name:
            print(f'  Auto-corrected supplier name: "{supplier_name}" → "{match}"')
        fixed['supplier_name'] = match
    else:
        errors.append(f'Supplier name "{supplier_name}" not recognised.\n    Valid options: {VALID_SUPPLIERS}')

    # Quantity
    try:
        qty = int(quantity_units)
        if qty <= 0:
            errors.append('Quantity must be a positive number.')
        else:
            fixed['quantity_units'] = qty
    except:
        errors.append(f'Quantity "{quantity_units}" is not a valid number.')

    # Order date
    try:
        pd.to_datetime(order_date)
        fixed['order_date'] = order_date
    except:
        errors.append(f'Order date "{order_date}" is not valid. Use format YYYY-MM-DD (e.g. 2025-07-10).')

    return fixed, errors

def fetch_forecast(city):
    lat, lon = CITY_COORDS[city]
    resp = requests.get('https://api.open-meteo.com/v1/forecast', params={
        'latitude': lat, 'longitude': lon,
        'daily': 'precipitation_sum,temperature_2m_max,wind_speed_10m_max',
        'forecast_days': 7, 'timezone': 'Asia/Kolkata'
    }, timeout=30)
    resp.raise_for_status()
    wf   = pd.DataFrame(resp.json()['daily'])
    rain = float(wf['precipitation_sum'].fillna(0).mean())
    temp = float(wf['temperature_2m_max'].fillna(28).mean())
    wind = float(wf['wind_speed_10m_max'].fillna(10).mean())
    sev  = 2 if (rain > 50 or wind > 60) else (1 if (rain > 15 or wind > 35) else 0)
    return rain, temp, wind, sev

def get_sentiment(city, month):
    row = sentiment_lookup[
        (sentiment_lookup['affected_city'] == city) &
        (sentiment_lookup['month'] == month)
    ]
    if len(row) == 0:
        return 0.0, 0.0
    return float(row['avg_sentiment'].values[0]), float(row['avg_disruption_signal'].values[0])

def get_supplier_risk(supplier_name):
    row = d4[d4['supplier_name'] == supplier_name]
    if len(row) == 0:
        return {c: float(d4[c].median()) for c in
                ['composite_risk_score','delivery_reliability_score',
                 'regional_risk_index','strike_incidents_3yr',
                 'financial_stability_score']} | {'risk_category': 'Medium'}
    return row[['composite_risk_score','delivery_reliability_score',
                'regional_risk_index','strike_incidents_3yr',
                'financial_stability_score','risk_category']].iloc[0].to_dict()

def get_top_causes(origin_city, transport_mode, weather_sev, sentiment, supplier_risk_cat):
    causes = []
    if weather_sev == 2:
        causes.append(f'Severe weather alert in {origin_city} (next 7 days)')
    elif weather_sev == 1:
        causes.append(f'Moderate rainfall/wind in {origin_city} (next 7 days)')
    if sentiment < -0.4:
        causes.append(f'Negative news sentiment in {origin_city} this month')
    if supplier_risk_cat in ('High', 'Medium'):
        causes.append(f'Supplier rated {supplier_risk_cat} risk category')
    if transport_mode == 'Road':
        causes.append('Road transport is vulnerable during weather events')
    if len(causes) == 0:
        causes.append('No major individual risk factor — combined features indicate risk')
    return causes[:3]

print('Helper functions with validation defined. ✅')

Helper functions with validation defined. ✅


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MAIN PREDICTION FUNCTION                                       ║
# ║  Flow: User Input → Weather API → Isolation Forest              ║
# ║        → LSTM → XGBoost → Full Report                           ║
# ╚══════════════════════════════════════════════════════════════════╝

def predict_shipment(origin_city, destination_city, product_category,
                     transport_mode, quantity_units, order_date, supplier_name):

    order_dt     = pd.to_datetime(order_date)
    shipment_month = order_dt.month

    # ── Step 1: Fetch live weather forecast ─────────────────────────
    print(f'Fetching 7-day weather forecast for {origin_city}...')
    rain, temp, wind, sev = fetch_forecast(origin_city)

    # ── Step 2: Run Isolation Forest on weather ──────────────────────
    weather_vec = np.array([[rain, temp, wind, sev]])
    iso_pred    = iso_forest.predict(weather_vec)
    weather_anomaly = int(iso_pred[0] == -1)   # 1 = anomaly

    # ── Step 3: Get sentiment from D3 lookup ─────────────────────────
    sentiment, disruption_signal = get_sentiment(origin_city, shipment_month)

    # ── Step 4: Get supplier risk from D4 ───────────────────────────
    risk = get_supplier_risk(supplier_name)

    # ── Step 5: Get city historical averages ─────────────────────────
    hist_count = city_hist_avg.get(origin_city, 0.0)
    cr = city_rolling.get(origin_city, {'avg_rolling7':0,'avg_rolling14':0,'avg_lag1':0,'avg_lag2':0})

    # ── Step 6: Encode categoricals ──────────────────────────────────
    transport_enc = int(le_transport.transform([transport_mode])[0])
    product_enc   = int(le_product.transform([product_category])[0])
    city_enc      = int(le_city.transform([origin_city])[0])
    dest_enc      = int(le_dest.transform([destination_city])[0])

    # ── Step 7: Estimate distance from D1 city pair average ──────────
    pair = merged[
        (merged['supplier_city'] == origin_city) &
        (merged['destination_city'] == destination_city)
    ]['distance_km']
    distance_km = float(pair.mean()) if len(pair) > 0 else float(merged['distance_km'].mean())

    # ── Step 8: Estimate promised delivery days from D1 average ──────
    promised_days_est = float(merged[
        (merged['transport_mode'] == transport_mode)
    ]['promised_delivery_days'].mean())

    # ── Step 9: Build feature dict ───────────────────────────────────
    feat = {
        'distance_km':                 distance_km,
        'promised_delivery_days':      promised_days_est,
        'quantity_units':              float(quantity_units),
        'rainfall_mm':                 rain,
        'temperature_celsius':         temp,
        'wind_speed_kmh':              wind,
        'severity_score':              float(sev),
        'weather_anomaly_score':       float(weather_anomaly),
        'sentiment_score':             sentiment,
        'disruption_signal':           disruption_signal,
        'composite_risk_score':        float(risk['composite_risk_score']),
        'delivery_reliability_score':  float(risk['delivery_reliability_score']),
        'regional_risk_index':         float(risk['regional_risk_index']),
        'strike_incidents_3yr':        float(risk['strike_incidents_3yr']),
        'financial_stability_score':   float(risk['financial_stability_score']),
        'historical_disruption_count': hist_count,
        'rolling_7d_delay':            cr['avg_rolling7'],
        'rolling_14d_delay':           cr['avg_rolling14'],
        'lag1_delay':                  cr['avg_lag1'],
        'lag2_delay':                  cr['avg_lag2'],
        'transport_mode_enc':          float(transport_enc),
        'product_category_enc':        float(product_enc),
        'supplier_city_enc':           float(city_enc),
        'destination_city_enc':        float(dest_enc),
    }

    # ── Step 10: LSTM prediction ─────────────────────────────────────
    lstm_vec = np.array([feat[f] for f in LSTM_FEATURES], dtype=np.float32)
    # Scale using saved scaler (scaler was fitted on LSTM_FEATURES only)
    lstm_vec_scaled = scaler.transform(lstm_vec.reshape(1, -1))
    lstm_seq  = np.tile(lstm_vec_scaled, (SEQ_LEN, 1)).reshape(1, SEQ_LEN, len(LSTM_FEATURES))
    lstm_prob = float(lstm_model.predict(lstm_seq, verbose=0)[0][0])

    # ── Step 11: XGBoost prediction ──────────────────────────────────
    feat['lstm_disruption_prob'] = lstm_prob
    xgb_df   = pd.DataFrame([[feat[f] for f in XGB_FEATURES]], columns=XGB_FEATURES)
    xgb_prob = float(xgb_model.predict_proba(xgb_df)[0][1])

    # ── Step 12: Final risk score ─────────────────────────────────────
    # XGBoost is the final model. LSTM already fed in as feature.
    final_prob = xgb_prob
    final_pct  = round(final_prob * 100, 1)

    if final_pct >= 70:
        risk_label = 'HIGH RISK ⚠️'
    elif final_pct >= 40:
        risk_label = 'MEDIUM RISK 🟡'
    else:
        risk_label = 'LOW RISK 🟢'

    # Model confidence = 1 - entropy-like measure
    confidence = round((abs(final_prob - 0.5) / 0.5) * 100, 1)

    # ── Step 13: Delivery date estimation ────────────────────────────
    base_travel_days = int(round(distance_km / 400))  # ~400 km/day for road
    if transport_mode == 'Air':      base_travel_days = max(1, int(round(distance_km / 800)))
    elif transport_mode == 'Rail':   base_travel_days = int(round(distance_km / 500))

    # Expected delay from historical city average, scaled by risk
    avg_delay = float(merged[merged['supplier_city'] == origin_city]['delay_days'].mean())
    predicted_delay = int(round(avg_delay * final_prob * 1.5))
    estimated_delivery = order_dt + pd.Timedelta(days=base_travel_days)
    expected_actual    = order_dt + pd.Timedelta(days=base_travel_days + predicted_delay)

    # ── Step 14: Top causes ──────────────────────────────────────────
    causes = get_top_causes(origin_city, transport_mode, sev, sentiment, risk['risk_category'])

    return {
        'origin_city':         origin_city,
        'destination_city':    destination_city,
        'product_category':    product_category,
        'transport_mode':      transport_mode,
        'order_date':          order_date,
        'supplier_name':       supplier_name,
        'distance_km':         round(distance_km),
        'base_travel_days':    base_travel_days,
        'estimated_delivery':  estimated_delivery.strftime('%Y-%m-%d'),
        'predicted_delay':     predicted_delay,
        'expected_actual':     expected_actual.strftime('%Y-%m-%d'),
        'lstm_prob':           round(lstm_prob * 100, 1),
        'xgb_prob':            round(xgb_prob * 100, 1),
        'final_risk_pct':      final_pct,
        'risk_label':          risk_label,
        'confidence':          confidence,
        'weather_sev':         sev,
        'rainfall_mm':         round(rain, 1),
        'wind_kmh':            round(wind, 1),
        'temp_c':              round(temp, 1),
        'weather_anomaly':     weather_anomaly,
        'sentiment':           round(sentiment, 3),
        'supplier_risk_cat':   risk['risk_category'],
        'composite_risk':      round(float(risk['composite_risk_score']), 3),
        'causes':              causes,
    }

print('predict_shipment() defined. ✅')

predict_shipment() defined. ✅


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ENTER YOUR SHIPMENT DETAILS HERE                               ║
# ║  Run this cell and the next cell to see the prediction report   ║
# ╚══════════════════════════════════════════════════════════════════╝

print('Enter your shipment details:')
print()

origin_city      = input('Enter origin city            : ').strip()
destination_city = input('Enter destination city       : ').strip()
product_category = input('Enter product category       : ').strip()
transport_mode   = input('Enter transport mode         : ').strip()
quantity_units   = input('Enter quantity (units)       : ').strip()
order_date       = input('Enter order date (YYYY-MM-DD): ').strip()
supplier_name    = input('Enter supplier name          : ').strip()

print('\nValidating inputs...')
fixed, errors = validate_inputs(origin_city, destination_city, product_category,
                                 transport_mode, quantity_units, order_date, supplier_name)

if errors:
    print('\n❌ Please fix the following before running prediction:\n')
    for e in errors:
        print(f'  • {e}')
else:
    print('\n✅ All inputs valid. Run the next cell to see prediction.')

Enter your shipment details:


Validating inputs...

✅ All inputs valid. Run the next cell to see prediction.


In [8]:
# Run prediction and print full report
result = predict_shipment(
    origin_city      = origin_city,
    destination_city = destination_city,
    product_category = product_category,
    transport_mode   = transport_mode,
    quantity_units   = quantity_units,
    order_date       = order_date,
    supplier_name    = supplier_name
)

sev_label = {0: 'Normal', 1: 'Moderate', 2: 'Severe / Heavy'}
anomaly_label = 'Yes — Unusual weather pattern detected' if result['weather_anomaly'] else 'No — Normal weather pattern'

print()
print('═'*55)
print('   SUPPLY CHAIN DISRUPTION PREDICTION REPORT')
print('═'*55)
print(f"  Origin City         : {result['origin_city']}")
print(f"  Destination City    : {result['destination_city']}")
print(f"  Product             : {result['product_category']}")
print(f"  Transport Mode      : {result['transport_mode']}")
print(f"  Order Date          : {result['order_date']}")
print(f"  Supplier            : {result['supplier_name']}")
print(f"  Estimated Distance  : {result['distance_km']} km")
print('─'*55)
print(f"  Estimated Travel Days     : {result['base_travel_days']} days")
print(f"  Predicted Delivery Date   : {result['estimated_delivery']}")
print(f"  Predicted Delay           : {result['predicted_delay']} days")
print(f"  Expected Actual Delivery  : {result['expected_actual']}")
print('─'*55)
print(f"  Disruption Risk Score     : {result['final_risk_pct']}%")
print(f"  Disruption Status         : {result['risk_label']}")
print(f"  Model Confidence          : {result['confidence']}%")
print('─'*55)
print('  Top Predicted Causes      :')
for i, c in enumerate(result['causes'], 1):
    print(f'    {i}. {c}')
print('─'*55)
print(f"  Weather (7-day forecast)  : Rainfall {result['rainfall_mm']} mm | Wind {result['wind_kmh']} km/h | Temp {result['temp_c']}°C")
print(f"  Weather Severity          : {sev_label[result['weather_sev']]} ({result['weather_sev']}/2)")
print(f"  Weather Anomaly Detected  : {anomaly_label}")
print(f"  Supplier Risk             : Composite Score {result['composite_risk']} | Category: {result['supplier_risk_cat']}")
print(f"  Sentiment Signal          : {result['sentiment']} (-1 very negative, +1 very positive)")
print('─'*55)
print(f"  LSTM Probability          : {result['lstm_prob']}%")
print(f"  XGBoost Final Probability : {result['xgb_prob']}%")
print('═'*55)

Fetching 7-day weather forecast for Mumbai...

═══════════════════════════════════════════════════════
   SUPPLY CHAIN DISRUPTION PREDICTION REPORT
═══════════════════════════════════════════════════════
  Origin City         : Mumbai
  Destination City    : Pune
  Product             : Textiles
  Transport Mode      : Road
  Order Date          : 2026-04-26
  Supplier            : Reliance Logistics
  Estimated Distance  : 1524 km
───────────────────────────────────────────────────────
  Estimated Travel Days     : 4 days
  Predicted Delivery Date   : 2026-04-30
  Predicted Delay           : 3 days
  Expected Actual Delivery  : 2026-05-03
───────────────────────────────────────────────────────
  Disruption Risk Score     : 27.0%
  Disruption Status         : LOW RISK 🟢
  Model Confidence          : 46.0%
───────────────────────────────────────────────────────
  Top Predicted Causes      :
    1. Negative news sentiment in Mumbai this month
    2. Road transport is vulnerable during we

In [9]:
# ── Optional: Quick test with hardcoded values (no input() needed) ────────────
# Uncomment and run this cell if you want to test without typing inputs

# result = predict_shipment(
#     origin_city      = 'Mumbai',
#     destination_city = 'Delhi',
#     product_category = 'Pharmaceuticals',
#     transport_mode   = 'Road',
#     quantity_units   = 500,
#     order_date       = '2025-07-10',
#     supplier_name    = 'Tata Suppliers Pvt Ltd'
# )
# print(result)